# x8_chuvaa — Precipitação Anual e Anomalias Climáticas CHIRPS v2

**Grupo no Modelo Econométrico:** `Geral`  
**Definição Operacional:** precipitação anual e anomalias (CHIRPS), mm ou escore padronizado, municipal, anual  
**Conceito e Mecanismo Econômico:** Desvios em relação à normal climatológica (z-score e anomalias em mm). Mede choques climáticos exógenos (secas ou excessos de chuva).  

---

### Origem dos Dados Brutos:
- `2_INTI\xdata\chap1\processed\tabular\h2__precip_anomaly__chirps_v2__ano_civil_candidatos.csv`

### Observações e Regras Específicas do Documento de Hipóteses:
> - Totais/anomalias em mm; z-scores adimensionais; comparar referências 1981–2010 e 1991–2020.
> - Ano civil e referências climáticas são candidatos; anomalia de chuva não é balanço hídrico.

### Salvaguarda Territorial (141 Municípios de Mato Grosso):
> **Regra Obrigatória do Projeto:** O estado de Mato Grosso é considerado com exatamente **141 municípios**.  
> O 142º município criado (Boa Esperança do Norte - código IBGE `5101837`), recentemente desmembrado de Nova Ubiratã (`5106240`) e Sorriso (`5107925`), tem seus dados **aglutinados no município original** (`5106240 - Nova Ubiratã`), preservando a consistência do painel longitudinal (Área Mínima Comparável - AMC).

---
## Protocolo Padronizado de 6 Fases (AED):
1. **Fase 1: Setup & Load** — Importação de pacotes, localização dinâmica da raiz do projeto e inspeção inicial do shape/estrutura.
2. **Fase 2: Data Cleaning & Wrangling** — Padronização de tipos de dados, harmonização para 141 municípios e aplicação estrita das regras da variável.
3. **Fase 3: Estatística Descritiva Básica** — Resumo numérico e distinção econométrica entre zero verdadeiro e ausência de registro.
4. **Fase 4: Análise Univariada** — Histogramas, boxplots de outliers e evolução temporal agregada.
5. **Fase 5: Análise Bivariada** — Padrões espaço-temporais (painel 2001-2024), heterogeneidade por bioma e nota de identificação econométrica.
6. **Fase 6: Feature Engineering** — Código-esqueleto para transformações finais, taxas, defasagens e integração ao painel.


## Fase 1: Setup & Load

Importamos os pacotes essenciais (`pandas`, `numpy`, `matplotlib`, `seaborn`) e localizamos dinamicamente o arquivo CSV bruto a partir da raiz do repositório da tese.

### Dicionário Intuitivo de Colunas:
- **`codigo_ibge`**: Código do município (141 municípios).
- **`ano`**: Ano.
- **`precip_mm_anual`**: Precipitação acumulada anual (mm).
- **`precip_anomaly_1981_2010`**: Anomalia frente à normal 1981-2010 (mm).
- **`precip_zscore_1981_2010`**: Z-score da chuva frente a 1981-2010 (adimensional).
- **`precip_anomaly_1991_2020`**: Anomalia frente a 1991-2020.
- **`precip_zscore_1991_2020`**: Z-score frente a 1991-2020.


In [ ]:
# =============================================================================
# FASE 1: SETUP & LOAD
# =============================================================================
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração visual para gráficos limpos e didáticos
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

# Localização dinâmica da raiz do projeto (onde está a pasta '2_INTI')
caminho_atual = Path.cwd().resolve()
raiz_projeto = None
for p in [caminho_atual] + list(caminho_atual.parents):
    if (p / "2_INTI" / "RESEARCH_PROTOCOL.md").is_file() or (p / "AGENTS.md").is_file():
        raiz_projeto = p
        break

if raiz_projeto is None:
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto (onde está 2_INTI/RESEARCH_PROTOCOL.md).")

# Caminho exato do arquivo conforme especificado no documento de hipóteses
caminho_csv = raiz_projeto / r"2_INTI\xdata\chap1\processed\tabular\h2__precip_anomaly__chirps_v2__ano_civil_candidatos.csv"

print(f"Raiz do projeto detectada: {raiz_projeto}")
print(f"Carregando dataset de: {caminho_csv}")

if not caminho_csv.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {caminho_csv}")

# Leitura com suporte a caracteres especiais (BOM / UTF-8)
df = pd.read_csv(caminho_csv, sep=',', encoding='utf-8-sig', low_memory=False)

print(f"Dimensões do dataset (linhas, colunas): {df.shape}")
print("\n--- Primeiras 5 linhas do dataset bruto ---")
df.head()

In [ ]:
print("--- Informações estruturais das colunas e tipos de dados ---")
df.info()

## Fase 2: Data Cleaning & Wrangling

Realizamos a limpeza sistemática dos dados respeitando as regras específicas para `x8_chuvaa`:
- **Atenção:** Totais/anomalias em mm; z-scores adimensionais; comparar referências 1981–2010 e 1991–2020.
- **Atenção:** Ano civil e referências climáticas são candidatos; anomalia de chuva não é balanço hídrico.
- **Salvaguarda Territorial:** Harmonização estrita para 141 municípios de Mato Grosso. Se houver dados de Boa Esperança do Norte (`5101837`), são aglutinados em Nova Ubiratã (`5106240`).


In [ ]:
# =============================================================================
# FASE 2: DATA CLEANING & WRANGLING
# =============================================================================
df.columns = df.columns.str.strip().str.lower().str.replace('\ufeff', '', regex=False)
df['codigo_ibge'] = df['codigo_ibge'].astype(str).str.zfill(7)
df['ano'] = pd.to_numeric(df['ano'], errors='coerce')

# HARMONIZAÇÃO TERRITORIAL (141 MUNICÍPIOS):
df['codigo_ibge'] = df['codigo_ibge'].replace({'5101837': '5106240'})
df = df[df['codigo_ibge'].str.startswith('51')].copy()

# REGRA DO TXT: 'Totais/anomalias em mm; z-scores adimensionais; comparar referências 1981–2010 e 1991–2020.'
df = df.groupby(['codigo_ibge', 'ano'], as_index=False)[['precip_mm_anual', 'precip_zscore_1981_2010', 'precip_zscore_1991_2020']].mean()
print(f"Municípios únicos: {df['codigo_ibge'].nunique()} (141 municípios)")

## Fase 3: Estatística Descritiva Básica

Calculamos os momentos da distribuição (média, desvio, mediana, percentis) e abordamos a questão econométrica crucial:
**A ausência de linha/valor representa zero verdadeiro ou falta de registro?**


In [ ]:
# =============================================================================
# FASE 3: ESTATÍSTICA DESCRITIVA BÁSICA
# =============================================================================
print("Descritiva dos Z-scores de Chuva (1981-2010):")
print(df['precip_zscore_1981_2010'].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]))

## Fase 4: Análise Univariada

Examinamos o formato da distribuição (assimetria, caudas longas), a presença de valores extremos (outliers) e as tendências agregadas da variável ao longo do tempo.


In [ ]:
# =============================================================================
# FASE 4: ANÁLISE UNIVARIADA
# =============================================================================
sns.histplot(df['precip_zscore_1981_2010'].dropna(), bins=30, kde=True, color='skyblue')
plt.title("Distribuição dos Z-scores de Precipitação Anual")
plt.axvline(0, color='red', linestyle='--')
plt.show()

## Fase 5: Análise Bivariada (Espaço-Temporal & Biomas)

Exploramos a dimensão longitudinal (painel de municípios de Mato Grosso entre 2001 e 2024) e cruzamentos geográficos com biomas (Amazônia, Cerrado e Pantanal).

> **Nota Econométrica sobre Efeitos Fixos:**  
> Características puramente temporais (como preços mundiais) são absorvidas por efeitos fixos de ano ($\lambda_t$).  
> Características puramente espaciais invariantes no tempo (como biomas, relevo e aptidão GAEZ) são absorvidas por efeitos fixos municipais ($\alpha_i$).  
> Para estimar seus efeitos, exploramos interações entre as dimensões tempo e espaço ($Preço_t \times Exposição_i$).


In [ ]:
# =============================================================================
# FASE 5: ANÁLISE BIVARIADA
# =============================================================================
df_painel = df[(df['ano'] >= 2001) & (df['ano'] <= 2024)]
sns.lineplot(data=df_painel, x='ano', y='precip_zscore_1981_2010', marker='o')
plt.title("Evolução das Anomalias de Chuva em MT (2001-2024)")
plt.axhline(0, color='grey', linestyle='--')
plt.show()

## Fase 6: Feature Engineering — Anomalias Padronizadas (Z-Score) e Choques Extremos

### 1. Step 1: Padronização Climatológica (Z-Score)
- **Fórmula:**
  $$Z_{i, t} = \frac{Chuva_{i, t} - \mu_{i, 1981-2010}}{\sigma_{i, 1981-2010}}$$
- O Z-score compara o volume de chuva de cada ano com o histórico normal de longo prazo do próprio município, eliminando diferenças de nível entre cidades chuvosas e secas.

### 2. Step 2: Dummies de Choques Climáticos Extremos
- **Dummy de Seca Severa:** $SecaSevera_{i, t} = 1$ se $Z_{i, t} < -1{,}5$ (ano extraordinariamente seco, que quebra a safra).
- **Dummy de Excesso de Chuva:** $ExcessoChuva_{i, t} = 1$ se $Z_{i, t} > 1{,}5$ (ano excessivamente chuvoso, que atrapalha a colheita e apodrece grãos).
- **Defasagem $t-1$:** O choque climático passado gera aperto de liquidez, inibindo novos investimentos em conversão de terra.


In [ ]:
# =============================================================================
# FASE 6: FEATURE ENGINEERING (TRATAMENTOS FINAIS E HARMONIZAÇÃO)
# =============================================================================
# 1. Estruturação dos Z-scores de Anomalia Climática
df_anom = df[['codigo_ibge', 'ano', 'precip_zscore_1981_2010']].copy()

# 2. Criação de Dummies de Choques Climáticos Extremos
# dummy_seca_severa: 1 se a chuva ficou mais de 1.5 desvios-padrão abaixo da média histórica do município
df_anom['dummy_seca_severa'] = (df_anom['precip_zscore_1981_2010'] < -1.5).astype(int)

# dummy_excesso_chuva: 1 se a chuva ficou mais de 1.5 desvios-padrão acima da média histórica
df_anom['dummy_excesso_chuva'] = (df_anom['precip_zscore_1981_2010'] > 1.5).astype(int)

# 3. Defasagem Temporal de 1 ano (t-1)
df_anom.sort_values(['codigo_ibge', 'ano'], inplace=True)
df_anom['dummy_seca_lag1'] = df_anom.groupby('codigo_ibge')['dummy_seca_severa'].shift(1)
df_anom['zscore_precip_lag1'] = df_anom.groupby('codigo_ibge')['precip_zscore_1981_2010'].shift(1)

painel_anom = df_anom[(df_anom['ano'] >= 2001) & (df_anom['ano'] <= 2024)].copy()

print("Choques Climáticos Extremos Identificados no Painel:")
print(f"Total de ocorrências de seca severa (Z < -1.5): {painel_anom['dummy_seca_severa'].sum()}")
print(f"Total de ocorrências de excesso de chuva (Z > 1.5): {painel_anom['dummy_excesso_chuva'].sum()}")
print(painel_anom[['codigo_ibge', 'ano', 'precip_zscore_1981_2010', 'dummy_seca_severa', 'dummy_seca_lag1']].head())